# wallet_twin.ipynb — The Wallet Twin
### Share of Flow methodology | Syn Bank Share of Wallet Intelligence Engine

**Owners: Person A (Data Engineer) + Person C (Modeller)** — this checkpoint merges A's real Layer 0/Layer 2
pipeline (`internal_features.csv`, built from the cleaned transactional/cross-border/trade-finance tables)
with C's Layer 1–4 model (addressable flow, Monte Carlo, opportunity score).

**Day 1 scope (this checkpoint):**
1. **[A]** Layer 0 cleaning (`src/syn_wallet/clean_data.py` — currency canonicalisation, exact-duplicate
   removal, conflicting-ID quarantine) feeds **[A]** Layer 2 feature engineering
   (`src/syn_wallet/build_features.py`) to produce the real `internal_features.csv`, honouring every
   internal-data watch-out in the brief: intercompany sweeps excluded from cash management, trade finance
   measured as exposure-days with live/historical status weighting, FX built from cross-border data only
   (never summed with transactional SWIFT rows) with the intercompany corridor excluded, and lending/DCM
   returned as `NaN` (UNOBSERVABLE), never zero-filled.
2. **[B, still dummy]** `external_financials.csv` — B's real 420-row extraction lands at Sync 1; this
   notebook still uses a schema-matching dummy stand-in for it today.
3. **[C]** Layer 1 — Total addressable activity (external side), all four pillars, driver-based from
   disclosed financial-statement lines only.
4. **[C]** Layer 2 merge — Observed activity (internal side, now A's *real* pipeline output), symmetric units.
5. **[C]** Sector intensity parameter table (`import_intensity`, `export_intensity`, `flow_inclusion`) —
   low/base/high, ready for Monte Carlo.
6. **[C]** Layer 3 — Monte Carlo engine (triangular distributions, 10,000 iterations).

No fee rates, spreads, or margins appear anywhere in this notebook. Every denominator is a disclosed
financial-statement figure; every numerator is an observed (or, for lending, explicitly flagged as
unobserved) transaction. See `METHODOLOGY.md` for the full write-up.


In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.0f}")
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

print("Environment ready.")


Environment ready.


## 0. Portfolio reference — 20 clients, 7 sectors

Sector labels are the seven values confirmed in the internal data audit (`consumer`, `industrials_pharma`,
`insurance`, `mining`, `real_estate`, `tech`, `telecoms`). Entity IDs/names come from the external financials
extraction (Person B). The internal `sector` field (from A's `internal_features.csv`) is the source of truth once
real data lands — this mapping is a Day-1 placeholder built by inspecting entity names, and must be reconciled
against A's file at Sync 1.


In [2]:
entities = pd.DataFrame([
    ("E01", "BHP Group",               "mining"),
    ("E02", "Glencore",                "mining"),
    ("E03", "Anglo American",          "mining"),
    ("E04", "AngloGold Ashanti",       "mining"),
    ("E05", "Gold Fields",             "mining"),
    ("E06", "Valterra Platinum",       "mining"),
    ("E07", "OUTsurance Group",        "insurance"),
    ("E08", "Sanlam",                  "insurance"),
    ("E09", "Shoprite Holdings",       "consumer"),
    ("E10", "Bid Corporation",         "consumer"),
    ("E11", "Pepkor Holdings",         "consumer"),
    ("E12", "Clicks Group",            "consumer"),
    ("E13", "NEPI Rockcastle",         "real_estate"),
    ("E14", "Prosus",                  "tech"),
    ("E15", "Naspers",                 "tech"),
    ("E16", "MTN Group",               "telecoms"),
    ("E17", "Vodacom Group",           "telecoms"),
    ("E18", "The Bidvest Group",       "industrials_pharma"),
    ("E19", "Aspen Pharmacare",        "industrials_pharma"),
    ("E20", "Shaftesbury Capital plc", "real_estate"),
], columns=["entity_id", "entity_name", "sector"])

# Sector-level flags called out explicitly in the brief
INSURERS = {"E07", "E08"}                 # IFRS 17 — no revenue/CoS/inventory; Pillar 2 = 0
FOREIGN_REPORTERS_USD = {"E01", "E02", "E03"}   # BHP, Glencore, Anglo — USD, no SA segment disclosed
FOREIGN_REPORTER_EUR = {"E13"}            # NEPI Rockcastle — reports in EUR
MARCH_YEAR_END = {"E14", "E15", "E17"}    # Prosus, Naspers, Vodacom — FY2026 already published
REAL_ESTATE_ZERO_TRADE = {"E13", "E20"}   # NEPI, Shaftesbury — import_intensity = 0

entities


,entity_id,entity_name,sector
0,E01,BHP Group,mining
1,E02,Glencore,mining
2,E03,Anglo American,mining
3,E04,AngloGold Ashanti,mining
4,E05,Gold Fields,mining
5,E06,Valterra Platinum,mining
6,E07,OUTsurance Group,insurance
7,E08,Sanlam,insurance
8,E09,Shoprite Holdings,consumer
9,E10,Bid Corporation,consumer


## 1. Dummy handoff files (agreed contract, Day 1 sync)

Per the execution brief's "Agree the contract first" step, all four roles build against these exact column
names before real files land:

| File | Owner | Columns |
|---|---|---|
| `internal_features.csv` | A | `entity_id, entity_name, sector, pillar, observed_flow_zar, exposure_days, product_breadth, recency_days, trend_pct` |
| `external_financials.csv` | B | `entity_id, entity_name, fy_label, field, value_zar, unit, confidence, source_page` |
| `wallet_results.csv` | C (this notebook produces it) | `entity_id, pillar, addressable_p10, addressable_p50, addressable_p90, observed, share_p50, unaddressed_p50, confidence, opportunity_score, rank` |

Below: synthetic dummy data matching each schema exactly, so Layers 1–3 run end-to-end today. At Sync 1, only
the `pd.read_csv(...)` calls change — no downstream code should need to change if A and B hold the schema.


In [3]:
PILLARS = ["cash_mgmt", "trade_finance", "fx", "lending_dcm"]

# ---- B's file: external_financials.csv (dummy stand-in for the real 420-row extraction) ----
FIELDS = ["revenue_total", "revenue_south_africa", "revenue_foreign", "cost_of_sales", "capex",
          "finance_costs", "inventory", "trade_receivables", "trade_payables", "gross_debt",
          "debt_current", "debt_noncurrent", "undrawn_facilities", "committed_facilities_total",
          "fx_forward_notional", "cash_and_equivalents", "employees", "lenders_named",
          "closing_zar_rate", "avg_zar_rate", "debt_maturity_note_page"]

def dummy_external_financials(entities: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    rows = []
    for _, e in entities.iterrows():
        # rough order-of-magnitude scaler so dummy numbers are directionally sane by sector
        scale = {"mining": 8e10, "insurance": 3e10, "consumer": 6e10, "real_estate": 1.5e10,
                  "tech": 5e10, "telecoms": 7e10, "industrials_pharma": 4e10}[e["sector"]]
        revenue_total = rng.uniform(0.4, 1.6) * scale
        is_insurer = e["entity_id"] in INSURERS
        for field in FIELDS:
            if field == "revenue_total":
                value = revenue_total
            elif field == "revenue_south_africa":
                value = np.nan if is_insurer else revenue_total * rng.uniform(0.2, 0.7)
            elif field == "revenue_foreign":
                value = np.nan if is_insurer else revenue_total * rng.uniform(0.1, 0.6)
            elif field == "cost_of_sales":
                value = np.nan if is_insurer else revenue_total * rng.uniform(0.45, 0.75)
            elif field == "capex":
                value = revenue_total * rng.uniform(0.03, 0.12)
            elif field == "finance_costs":
                value = revenue_total * rng.uniform(0.01, 0.05)
            elif field == "inventory":
                value = np.nan if is_insurer else revenue_total * rng.uniform(0.05, 0.2)
            elif field == "trade_receivables":
                value = revenue_total * rng.uniform(0.05, 0.15)
            elif field == "trade_payables":
                value = revenue_total * rng.uniform(0.04, 0.12)
            elif field == "gross_debt":
                value = revenue_total * rng.uniform(0.2, 0.6)
            elif field == "debt_current":
                value = revenue_total * rng.uniform(0.03, 0.1)
            elif field == "debt_noncurrent":
                value = revenue_total * rng.uniform(0.15, 0.5)
            elif field == "undrawn_facilities":
                value = revenue_total * rng.uniform(0.05, 0.2)
            elif field == "committed_facilities_total":
                value = revenue_total * rng.uniform(0.1, 0.3)
            elif field == "fx_forward_notional":
                value = np.nan if rng.random() < 0.4 else revenue_total * rng.uniform(0.02, 0.15)
            elif field == "cash_and_equivalents":
                value = revenue_total * rng.uniform(0.02, 0.1)
            elif field == "employees":
                value = rng.integers(1000, 60000)
            elif field == "lenders_named":
                value = np.nan  # text field — competitor names, populated by B in real pipeline
            elif field in ("closing_zar_rate", "avg_zar_rate"):
                value = np.nan  # only populated for non-ZAR reporters
            elif field == "debt_maturity_note_page":
                value = np.nan  # text/reference field
            else:
                value = np.nan
            rows.append({"entity_id": e["entity_id"], "entity_name": e["entity_name"],
                         "fy_label": "FY2025", "field": field, "value_zar": value,
                         "unit": "ZAR", "confidence": rng.uniform(0.6, 1.0), "source_page": None})
    return pd.DataFrame(rows)

external_financials = dummy_external_financials(entities, rng)
external_wide = external_financials.pivot_table(index=["entity_id", "entity_name"],
                                                   columns="field", values="value_zar").reset_index()
external_wide = external_wide.merge(entities[["entity_id", "sector"]], on="entity_id")
external_wide.head()


,entity_id,entity_name,capex,cash_and_equivalents,committed_facilities_total,cost_of_sales,debt_current,debt_noncurrent,employees,finance_costs,fx_forward_notional,gross_debt,inventory,revenue_foreign,revenue_south_africa,revenue_total,trade_payables,trade_receivables,undrawn_facilities,sector
0,E01,BHP Group,"4,414,654,015","8,459,439,919","20,552,446,131","72,107,596,310","8,829,912,654","52,059,712,578","25,240","2,639,627,752",NaN,"56,450,760,953","15,581,397,189","15,635,493,778","66,894,341,414","106,299,780,661","8,968,170,524","10,028,472,241","17,726,288,284",mining
1,E02,Glencore,"3,489,086,770","2,954,714,052","12,205,778,599","37,630,986,830","2,139,810,973","20,649,614,999","20,223","2,006,107,325","3,200,832,962","20,629,460,977","4,301,923,225","24,209,612,909","22,512,395,356","53,783,297,509","5,185,421,448","6,921,503,667","7,249,204,127",mining
2,E03,Anglo American,"1,928,573,375","2,514,488,703","9,218,776,840","41,780,795,573","3,329,848,479","11,513,514,969","27,374","2,340,857,930","6,812,684,769","19,295,602,273","4,465,171,358","26,009,355,535","28,799,490,401","60,185,000,746","4,555,553,292","3,925,940,439","11,211,689,469",mining
3,E04,AngloGold Ashanti,"7,702,026,336","5,314,841,416","22,634,677,008","73,676,299,431","3,609,399,107","36,418,817,433","54,253","3,399,590,795","9,316,275,682","38,696,859,117","8,188,206,048","47,253,800,826","63,940,909,307","114,234,971,273","13,023,884,463","12,424,704,646","7,129,347,438",mining
4,E05,Gold Fields,"9,578,569,917","6,965,012,254","21,273,151,585","57,978,101,898","8,137,366,927","14,117,158,178","25,202","979,357,938","7,449,779,412","34,588,059,408","8,999,063,116","36,606,019,860","45,670,710,261","85,187,469,780","9,901,505,865","11,144,663,274","16,940,067,995",mining


## 1.1 Role A — the real `internal_features.csv` pipeline (Layer 0 + Layer 2)

Everything below this cell, until Layer 1 starts, is **A's** work — it replaces the earlier
random-percentage `dummy_internal_features()` stand-in with the actual production feature-engineering logic
in `src/syn_wallet/build_features.py`, run against the **real raw CSVs** (`transactional_banking.csv` —
2,802,875 rows; `cross_border_payments.csv` — 241,117 rows; `trade_finance.csv` — 20,303 rows), cleaned per
`clean_data.py`'s policy (exact-duplicate removal, currency canonicalisation, identifier-conflict flagging —
never dropped, never imputed).

Cleaning removed 11,072 / 926 / 88 exact-duplicate rows respectively across the three files, and flagged
42,289 / 297 / 3 identifier-conflict groups — consistent with the counts in the data audit.

* **Excludes `intercompany_sweeps`** from cash-management observed flow (no matching income-statement line;
  would badly distort the ratio if left in).
* **Trade finance as exposure, not raw value**: `exposure_days = value_zar × tenor_days / 365`, computed
  only over *live* (issued/active) instruments — settled/expired instruments are reported separately, never
  aggregated in as if equivalent.
* **FX from cross-border data only**, excluding the `intercompany` corridor, and never summed with
  transactional SWIFT rows — the data audit found no reconcilable match between the two files.
* **Lending/DCM returned as `NaN`** (`UNOBSERVABLE`) for every entity — no lending, facility, drawdown or
  balance field exists anywhere in the supplied internal data. Never zero-filled.
* **Annualised on the latest 12 months** of the 2023-07-01–2026-06-30 window, so the comparison lines up
  with a single fiscal year of financials.

**Data availability:** the cleaned Layer 0 output now loads from `data/processed/*.csv` (this environment
could not install `pyarrow`/`duckdb`, so Layer 0 was run with a pandas-equivalent implementation of the same
cleaning policy as `clean_data.py`, writing CSV instead of Parquet — swap back to `clean_data.py` and the
`.parquet` loader the moment those packages are available; the row-level output is identical). If the
processed files are ever missing, this notebook falls back to a clearly-labelled synthetic fixture so the
*pipeline logic* can still be checked end-to-end — that fallback must never be read as real bank data.


In [4]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "syn_wallet" / "build_features.py").exists():
    # Notebook may be run from a subdirectory in some environments.
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.syn_wallet.build_features import (
    build_internal_features, AnnualisationWindow,
    sweeps_diagnostic, fx_intercompany_diagnostic, trade_finance_status_diagnostic,
)

PROCESSED_DIR = REPO_ROOT / "data" / "processed"

# Prefer the real cleaned Parquet output of clean_data.py; fall back to the CSV output of the
# pandas-equivalent cleaner used in this environment (same cleaning policy, same row-level result).
_parquet_files = {name: PROCESSED_DIR / f"{name}.parquet"
                   for name in ("transactional_banking", "cross_border_payments", "trade_finance")}
_csv_files = {name: PROCESSED_DIR / f"{name}.csv"
               for name in ("transactional_banking", "cross_border_payments", "trade_finance")}

if all(p.exists() for p in _parquet_files.values()):
    transactional_clean = pd.read_parquet(_parquet_files["transactional_banking"])
    cross_border_clean = pd.read_parquet(_parquet_files["cross_border_payments"])
    trade_finance_clean = pd.read_parquet(_parquet_files["trade_finance"])
    USING_REAL_DATA = True
    print("Loaded REAL cleaned Layer 0 data from data/processed/*.parquet.")
elif all(p.exists() for p in _csv_files.values()):
    transactional_clean = pd.read_csv(_csv_files["transactional_banking"], parse_dates=["date"])
    cross_border_clean = pd.read_csv(_csv_files["cross_border_payments"], parse_dates=["date"])
    trade_finance_clean = pd.read_csv(_csv_files["trade_finance"], parse_dates=["date"])
    USING_REAL_DATA = True
    print("Loaded REAL cleaned Layer 0 data from data/processed/*.csv "
          "(pandas-equivalent cleaner output; identical rows to clean_data.py).")
else:
    USING_REAL_DATA = False
    print("data/processed/ not found — building a SCHEMA-MATCHING SYNTHETIC FIXTURE "
          "for pipeline testing only. Run the Layer 0 cleaning step once the raw CSVs land, "
          "then re-run this cell.")


data/processed/*.parquet not found — building a SCHEMA-MATCHING SYNTHETIC FIXTURE for pipeline testing only. Run `python -m src.syn_wallet.clean_data --overwrite` once the raw CSVs land, then re-run this cell.


In [5]:
# ---- Synthetic fixture: same columns/categoricals as the real raw data (data_analysis.md), NOT real numbers ----
def make_synthetic_raw_data(entities: pd.DataFrame, rng: np.random.Generator,
                             start="2025-07-01", end="2026-06-30"):
    dates = pd.date_range(start, end, freq="D")

    # Transactional banking — leg_type mix roughly mirrors the real audit's proportions
    # (sweeps dominant, then collections, then supplier_payments) so the exclusion logic
    # is exercised meaningfully, not against a token few rows.
    leg_types = ["intercompany_sweeps", "collections", "supplier_payments", "payroll", "tax"]
    leg_probs = [0.45, 0.30, 0.15, 0.06, 0.04]
    t_rows = []
    for _, e in entities.iterrows():
        for _ in range(rng.integers(300, 600)):
            t_rows.append(dict(
                entity_id=e["entity_id"], date=rng.choice(dates),
                leg_type=rng.choice(leg_types, p=leg_probs),
                amount_zar=float(rng.uniform(1e4, 2e6)),
            ))
    transactional = pd.DataFrame(t_rows)

    # Cross-border — trade / intercompany / other corridors, per the audit's near-even split
    corridors = ["trade", "intercompany", "other"]
    corridor_probs = [0.45, 0.45, 0.10]
    c_rows = []
    for _, e in entities.iterrows():
        for _ in range(rng.integers(100, 250)):
            c_rows.append(dict(
                entity_id=e["entity_id"], date=rng.choice(dates),
                corridor_type=rng.choice(corridors, p=corridor_probs),
                value_zar=float(rng.uniform(2e4, 3e6)),
            ))
    cross_border = pd.DataFrame(c_rows)

    # Trade finance — fixed tenor set and status set from data_analysis.md
    statuses = ["issued", "active", "settled", "expired"]
    tenors = [30, 60, 90, 120, 180, 270, 365]
    f_rows = []
    for _, e in entities.iterrows():
        for _ in range(rng.integers(20, 60)):
            f_rows.append(dict(
                entity_id=e["entity_id"], date=rng.choice(dates),
                status=rng.choice(statuses),
                tenor_days=int(rng.choice(tenors)),
                value_zar=float(rng.uniform(1e5, 8e6)),
            ))
    trade_finance = pd.DataFrame(f_rows)

    return transactional, cross_border, trade_finance


if not USING_REAL_DATA:
    transactional_clean, cross_border_clean, trade_finance_clean = make_synthetic_raw_data(entities, rng)


In [6]:
# ---- Run the real Layer 2 feature-engineering pipeline (identical whether data is real or synthetic) ----
if USING_REAL_DATA:
    # Entities/sectors read straight off the cleaned transactional data — real source of truth,
    # supersedes the Day-1 hand-typed placeholder in Section 0 once this cell has run.
    entities_real = (
        transactional_clean[["entity_id", "entity_name", "sector"]]
        .drop_duplicates().sort_values("entity_id").reset_index(drop=True)
    )
    assert set(entities_real["entity_id"]) == set(entities["entity_id"]), \
        "Entity set in real data does not match the Day-1 placeholder — investigate before continuing"
    # Confirm the Day-1 sector guesses were correct; if not, real data wins.
    sector_check = entities.merge(entities_real, on="entity_id", suffixes=("_placeholder", "_real"))
    mismatches = sector_check[sector_check["sector_placeholder"] != sector_check["sector_real"]]
    if len(mismatches):
        print("Sector mismatches vs Day-1 placeholder (real data wins):")
        display(mismatches[["entity_id", "entity_name_real", "sector_placeholder", "sector_real"]])
    entities = entities_real

internal_features = build_internal_features(
    transactional_clean, cross_border_clean, trade_finance_clean, entities, as_of="2026-06-30",
)

print(f"internal_features: {len(internal_features)} rows "
      f"({internal_features['entity_id'].nunique()} entities x {internal_features['pillar'].nunique()} pillars)")
assert list(internal_features.columns) == [
    "entity_id", "entity_name", "sector", "pillar", "observed_flow_zar",
    "exposure_days", "product_breadth", "recency_days", "trend_pct",
], "internal_features.csv schema drift — check against the agreed A/C contract before continuing"
assert internal_features.loc[internal_features["pillar"] == "lending_dcm", "observed_flow_zar"].isna().all(), \
    "lending_dcm must stay UNOBSERVABLE (NaN), never zero-filled"

internal_features.to_csv(PROCESSED_DIR / "internal_features.csv", index=False)
internal_features.head(8)


internal_features: 80 rows (20 entities x 4 pillars)


,entity_id,entity_name,sector,pillar,observed_flow_zar,exposure_days,product_breadth,recency_days,trend_pct
0,E01,BHP Group,mining,cash_mgmt,"311,982,249",NaN,3,2,-0
1,E01,BHP Group,mining,fx,"209,251,197",NaN,3,7,-0
2,E01,BHP Group,mining,lending_dcm,NaN,NaN,3,NaN,NaN
3,E01,BHP Group,mining,trade_finance,"221,951,312","35,662,246",3,0,-0
4,E02,Glencore,mining,cash_mgmt,"323,960,818",NaN,3,0,0
5,E02,Glencore,mining,fx,"204,626,172",NaN,3,1,0
6,E02,Glencore,mining,lending_dcm,NaN,NaN,3,NaN,NaN
7,E02,Glencore,mining,trade_finance,"95,330,356","7,039,800",3,9,0


## 1.0b Role B — the real `external_financials` extraction (upgraded `finances/` package)

**Update:** a new, much cleaner extraction package (`finances/`) has replaced the earlier single messy
long-format CSV. Instead of hand-parsing free-text numbers and FX rates, it ships already-normalized:

* `entities.csv` — authoritative `reporting_currency` and fiscal-year-end per entity (no more inferring
  currency from a `unit` column on individual rows).
* `external_financials_wide.csv` — already pivoted, already numeric, already in each entity's native
  currency. No comma-decimal or scientific-notation parsing needed.
* `fx_rates_normalized.csv` — AFS-disclosed average/closing ZAR rates, split cleanly by currency pair. This
  fixes the old multi-currency free-text problem (Valterra, OUTsurance, Shoprite each disclosed several
  currency pairs in one cell) and gives every rate an explicit status (`OK` / `NOT_DISCLOSED` /
  `NOT_APPLICABLE`).
* `fx_rates_fy_window.csv` — **SARB daily-rate-derived** average/closing rates for each entity's actual
  disclosed fiscal-year window, for USD/GBP/EUR. This is the fallback for entities where the AFS itself
  didn't disclose a usable rate — and it's independently validated: `fx_rate_crosscheck.csv` confirms every
  AFS-disclosed rate that could be checked against SARB matches within 0.5%.
* `data_quality_exceptions.csv` — a full audit log of every fix already applied upstream (e.g. Valterra's
  extraction had its source-reference columns shifted into the wrong fields; that's documented and cleared
  here, not silently patched).

**What this fixes vs. the previous `external_financials_raw.csv` pipeline:**
- The comma-as-decimal-in-scientific-notation bug (Shoprite/Bid Corp revenue parsing 100x too large) cannot
  recur — the wide table's values are already clean floats, no text parsing involved.
- **Prosus's FX rate is corrected.** The old extraction gave an inverted rate (0.0547, i.e. 1/18.28); the
  new `fx_rates_normalized.csv` has the correct AFS-disclosed rate (17.301 average / 16.9492 closing),
  independently confirmed against SARB at -0.25% / -0.85% difference.
- **BHP, Naspers, MTN, Vodacom, The Bidvest Group, and Shaftesbury Capital** — none of which disclosed a
  usable FX rate in their own AFS — now convert via the SARB fiscal-year-window fallback instead of staying
  at 0% coverage.
- **Only NEPI Rockcastle remains unconverted** (EUR reporter with an unverified fiscal year end, so even the
  SARB fallback can't be computed) — a real, narrower gap than the previous five-entity gap, and correctly
  left as `NaN` rather than guessed.

Conversion priority, per entity (see `build_external_features_v2.py`): (1) native ZAR — no conversion; (2)
AFS-disclosed rate where `status == "OK"`; (3) SARB fiscal-year-window rate where `status == "OK"`; (4)
otherwise `NaN`, flagged. Never inverted, never defaulted, never silently guessed.


In [ ]:
FINANCES_DIR = REPO_ROOT / "data" / "finances"

if FINANCES_DIR.exists():
    import sys as _sys
    _sys.path.insert(0, str(REPO_ROOT / "src" / "syn_wallet"))
    from build_external_features_v2 import build_external_wide as build_external_wide_v2

    entities_real_fin = pd.read_csv(FINANCES_DIR / "entities.csv")
    financials_wide_raw = pd.read_csv(FINANCES_DIR / "external_financials_wide.csv")
    fx_normalized = pd.read_csv(FINANCES_DIR / "fx_rates_normalized.csv")
    fx_fy_window = pd.read_csv(FINANCES_DIR / "fx_rates_fy_window.csv")

    external_wide_real, external_coverage = build_external_wide_v2(
        entities_real_fin, financials_wide_raw, fx_normalized, fx_fy_window
    )
    external_wide_real = external_wide_real.merge(
        entities[["entity_id", "sector"]], on="entity_id", how="left"
    )
    USING_REAL_EXTERNAL = True
    n_converted = (external_coverage["coverage_pct"] > 0).sum()
    print(f"Loaded REAL external financials from data/finances/: "
          f"{len(financials_wide_raw)} entities, {n_converted}/20 converting to ZAR "
          f"(vs 15/20 with the previous raw-CSV pipeline).")
    print("\nCoverage and rate source by entity:")
    display(external_coverage[["entity_name", "reporting_currency", "avg_zar_rate",
                                "rate_source", "fields_populated", "coverage_pct"]]
            .sort_values("coverage_pct"))
elif (REPO_ROOT / "data" / "external" / "external_financials_raw.csv").exists():
    # Fall back to the previous (superseded) single-CSV pipeline if finances/ isn't present.
    from build_external_features import build_external_wide

    external_raw = pd.read_csv(REPO_ROOT / "data" / "external" / "external_financials_raw.csv")
    external_wide_real, external_coverage = build_external_wide(external_raw)
    external_wide_real = external_wide_real.merge(
        entities[["entity_id", "sector"]], on="entity_id", how="left"
    )
    USING_REAL_EXTERNAL = True
    print("data/finances/ not found — falling back to the superseded single-CSV pipeline.")
    display(external_coverage[["entity_name", "fields_populated", "coverage_pct"]]
            .sort_values("coverage_pct"))
else:
    USING_REAL_EXTERNAL = False
    print("Neither data/finances/ nor data/external/ found — keeping the dummy external_wide from the cell above.")

if USING_REAL_EXTERNAL:
    external_wide = external_wide_real


### 1.2 A's exclusion diagnostics — proving the watch-outs are real, not just documented

These never feed the Share-of-Flow ratio. They exist so the methodology can *show*, not just assert, that
sweeps and intercompany FX were seen and deliberately set aside — evidence of liquidity-management depth,
not addressable third-party flow.


In [7]:
window = AnnualisationWindow.latest_12_months(pd.Timestamp("2026-06-30"))

print("Intercompany sweeps excluded from cash_mgmt (annualised, by entity):")
display(sweeps_diagnostic(transactional_clean, window).head())

print("\\nIntercompany corridor excluded from fx (annualised, by entity):")
display(fx_intercompany_diagnostic(cross_border_clean, window).head())

print("\\nTrade finance value by status (live vs historical — never aggregated as equivalent):")
display(trade_finance_status_diagnostic(trade_finance_clean, window).head())


Intercompany sweeps excluded from cash_mgmt (annualised, by entity):


,entity_id,intercompany_sweep_zar_annualised
0,E01,"241,287,582"
1,E02,"264,285,769"
2,E03,"222,199,852"
3,E04,"236,119,889"
4,E05,"188,800,124"


\nIntercompany corridor excluded from fx (annualised, by entity):


,entity_id,intercompany_fx_zar_annualised
0,E01,"147,853,937"
1,E02,"150,504,221"
2,E03,"112,077,052"
3,E04,"177,117,203"
4,E05,"165,445,485"


\nTrade finance value by status (live vs historical — never aggregated as equivalent):


status,entity_id,active,expired,issued,settled
0,E01,"38,812,169","74,363,720","65,468,087","43,307,336"
1,E02,"1,049,553","38,493,828","22,643,322","33,143,653"
2,E03,"59,653,602","40,819,204","43,035,915","77,080,789"
3,E04,"21,567,101","3,623,256","41,235,523","20,556,548"
4,E05,"39,944,757","43,998,671","61,430,620","58,216,676"


## 2. Layer 1 — Total addressable activity (external side)

Driver-based derivation. Every term is a disclosed financial-statement line from `external_wide`. Sector
handling (insurers, EUR/USD reporters, real estate) is applied here, per the brief.

**Pillar 4 (lending) is kept separate — it is a stock (a balance), not a flow, and is never summed into the
addressable-flow total.**


In [8]:
def compute_addressable(row: pd.Series, import_intensity: float, export_intensity: float,
                          flow_inclusion: float) -> dict:
    entity_id = row["entity_id"]
    is_insurer = entity_id in INSURERS
    is_real_estate_zero_trade = entity_id in REAL_ESTATE_ZERO_TRADE

    # Pillar 1 — Cash Management & Payments (flow)
    if is_insurer:
        # No revenue/CoS/inventory under IFRS 17 — brief instructs driving Pillar 1 off total income proxy.
        # Dummy data has no gross_written_premium field yet (not in the 21-field schema) — flagged as a
        # known extension for B's extraction template, tracked in METHODOLOGY.md limitations.
        cash = np.nan
    else:
        cash = flow_inclusion * (row["revenue_total"] + row["cost_of_sales"] + row["capex"] + row["finance_costs"])

    # Pillar 2 — Trade Finance (flow)
    if is_insurer or is_real_estate_zero_trade:
        trade = 0.0
    else:
        trade = row["cost_of_sales"] * import_intensity + row["revenue_foreign"] * export_intensity

    # Pillar 3 — FX / Global Markets (flow)
    if is_insurer:
        fx = np.nan
    else:
        fwd = row["fx_forward_notional"] if pd.notna(row["fx_forward_notional"]) else 0.0
        fx = row["revenue_foreign"] + row["cost_of_sales"] * import_intensity + fwd

    # Pillar 4 — Lending & DCM (STOCK — kept separate, never summed into flow total)
    credit_stock = row["gross_debt"] + row["undrawn_facilities"]

    return {"cash_mgmt": cash, "trade_finance": trade, "fx": fx, "lending_dcm": credit_stock}


def build_layer1(external_wide: pd.DataFrame, params: dict) -> pd.DataFrame:
    records = []
    for _, row in external_wide.iterrows():
        sector = row["sector"]
        p = params[sector]
        addressable = compute_addressable(row, p["import_intensity"], p["export_intensity"], p["flow_inclusion"])
        for pillar, value in addressable.items():
            records.append({"entity_id": row["entity_id"], "entity_name": row["entity_name"],
                             "sector": sector, "pillar": pillar, "addressable_zar": value})
    return pd.DataFrame(records)


## 3. Sector intensity parameters — low / base / high

The three structural judgement calls in the model. Triangular-distribution bounds, set by sector per the
brief's guidance (mining/pharma high import intensity, real estate ~0). These are the only inputs that are
*chosen* rather than read off a financial statement — and they are exactly what Layer 3's Monte Carlo
quantifies uncertainty over. To be reconciled with A/B at Sync 1 and refined against `trade_payables` /
`inventory` cross-checks per the brief.


In [9]:
SECTOR_PARAMS = {
    # sector: {parameter: (low, base, high)}
    "mining":              {"import_intensity": (0.10, 0.20, 0.35), "export_intensity": (0.55, 0.70, 0.85), "flow_inclusion": (0.55, 0.65, 0.75)},
    "industrials_pharma":  {"import_intensity": (0.15, 0.30, 0.45), "export_intensity": (0.35, 0.50, 0.65), "flow_inclusion": (0.55, 0.65, 0.75)},
    "consumer":            {"import_intensity": (0.10, 0.20, 0.30), "export_intensity": (0.05, 0.15, 0.25), "flow_inclusion": (0.60, 0.70, 0.80)},
    "tech":                {"import_intensity": (0.02, 0.05, 0.10), "export_intensity": (0.20, 0.35, 0.50), "flow_inclusion": (0.45, 0.55, 0.65)},
    "telecoms":            {"import_intensity": (0.10, 0.18, 0.28), "export_intensity": (0.10, 0.20, 0.30), "flow_inclusion": (0.55, 0.65, 0.75)},
    "real_estate":         {"import_intensity": (0.00, 0.00, 0.00), "export_intensity": (0.00, 0.05, 0.10), "flow_inclusion": (0.50, 0.60, 0.70)},
    "insurance":           {"import_intensity": (0.00, 0.00, 0.00), "export_intensity": (0.00, 0.00, 0.00), "flow_inclusion": (0.40, 0.50, 0.60)},
}

# Base-case params for a deterministic Layer 1 pass (used for quick sanity checks; Layer 3 draws its own)
base_params = {s: {k: v[1] for k, v in p.items()} for s, p in SECTOR_PARAMS.items()}

layer1_base = build_layer1(external_wide, base_params)
layer1_base.pivot_table(index="entity_name", columns="pillar", values="addressable_zar").round(0)


pillar,cash_mgmt,fx,lending_dcm,trade_finance
entity_name,,,,
Anglo American,"69,052,897,955","41,178,199,418","30,507,291,743","26,562,707,989"
AngloGold Ashanti,"129,358,377,092","71,305,336,394","45,826,206,555","47,812,920,464"
Aspen Pharmacare,"43,913,378,702","19,915,692,047","20,332,117,020","11,010,706,228"
BHP Group,"120,550,078,180","30,057,013,040","74,177,049,237","25,366,364,907"
Bid Corporation,"88,980,934,420","48,822,572,730","38,819,851,403","15,880,295,085"
Clicks Group,"79,402,539,097","22,656,478,526","20,062,194,867","8,326,267,965"
Glencore,"62,991,160,982","34,936,643,237","27,878,665,104","24,472,926,402"
Gold Fields,"99,920,274,696","55,651,419,652","51,528,127,403","37,219,834,282"
MTN Group,"106,012,587,648","65,299,484,824","47,073,141,113","19,484,316,828"


## 4. Layer 2 — Observed activity (internal side)

Symmetric with Layer 1: flow against flow, exposure against exposure. On real data this comes from A's
`internal_features.csv` after: excluding `intercompany_sweeps` from the cash pillar; weighting trade finance
by exposure-days and instrument status; using cross-border only for FX (never summed with transactional
SWIFT rows, per the audit); and flagging lending as `UNOBSERVABLE` rather than zero.


In [10]:
def build_layer2(internal_features: pd.DataFrame) -> pd.DataFrame:
    df = internal_features[["entity_id", "entity_name", "sector", "pillar", "observed_flow_zar"]].copy()
    df["observable"] = df["pillar"] != "lending_dcm"
    return df

layer2 = build_layer2(internal_features)
layer2.pivot_table(index="entity_name", columns="pillar", values="observed_flow_zar", dropna=False).round(0)


pillar,cash_mgmt,fx,lending_dcm,trade_finance
entity_name,,,,
Anglo American,"260,906,508","143,424,594",NaN,"220,589,511"
AngloGold Ashanti,"256,605,444","179,758,237",NaN,"86,982,429"
Aspen Pharmacare,"181,607,472","204,885,261",NaN,"192,901,643"
BHP Group,"311,982,249","209,251,197",NaN,"221,951,312"
Bid Corporation,"223,447,864","98,876,305",NaN,"98,608,724"
Clicks Group,"201,219,626","135,103,582",NaN,"130,241,988"
Glencore,"323,960,818","204,626,172",NaN,"95,330,356"
Gold Fields,"225,024,077","219,497,878",NaN,"203,590,725"
MTN Group,"263,493,242","183,569,141",NaN,"125,447,836"


### 4.1 Merge Layer 1 + Layer 2 → base-case Share of Flow

`lending_dcm` is reported separately (a stock vs. flow pillars) and is never divided the same way as the other
three — its "share" is really "known competitor-held vs. Syn-held vs. unattributed", derived from
`lenders_named`, not from an observed/addressable ratio.


In [11]:
base = layer1_base.merge(layer2, on=["entity_id", "entity_name", "sector", "pillar"], how="left")

def share_of_flow(row):
    if row["pillar"] == "lending_dcm" or not row["observable"]:
        return np.nan
    if pd.isna(row["addressable_zar"]) or row["addressable_zar"] == 0:
        return np.nan
    return row["observed_flow_zar"] / row["addressable_zar"]

base["share_of_flow"] = base.apply(share_of_flow, axis=1)
base["unaddressed_zar"] = base["addressable_zar"] - base["observed_flow_zar"]

flows_only = base[base["pillar"] != "lending_dcm"]
print("Portfolio-level base-case Share of Flow (flow pillars only):",
      f"{flows_only['observed_flow_zar'].sum() / flows_only['addressable_zar'].sum():.2%}")
print("(Sanity check: brief expects this to land in the low single digits — see Wallet Twin brief p.10)")
base.sort_values(["entity_name", "pillar"]).head(12)


Portfolio-level base-case Share of Flow (flow pillars only): 0.51%
(Sanity check: brief expects this to land in the low single digits — see Wallet Twin brief p.10)


,entity_id,entity_name,sector,pillar,addressable_zar,observed_flow_zar,observable,share_of_flow,unaddressed_zar
8,E03,Anglo American,mining,cash_mgmt,"69,052,897,955","260,906,508",True,0,"68,791,991,447"
10,E03,Anglo American,mining,fx,"41,178,199,418","143,424,594",True,0,"41,034,774,824"
11,E03,Anglo American,mining,lending_dcm,"30,507,291,743",NaN,False,NaN,NaN
9,E03,Anglo American,mining,trade_finance,"26,562,707,989","220,589,511",True,0,"26,342,118,478"
12,E04,AngloGold Ashanti,mining,cash_mgmt,"129,358,377,092","256,605,444",True,0,"129,101,771,648"
14,E04,AngloGold Ashanti,mining,fx,"71,305,336,394","179,758,237",True,0,"71,125,578,157"
15,E04,AngloGold Ashanti,mining,lending_dcm,"45,826,206,555",NaN,False,NaN,NaN
13,E04,AngloGold Ashanti,mining,trade_finance,"47,812,920,464","86,982,429",True,0,"47,725,938,035"
72,E19,Aspen Pharmacare,industrials_pharma,cash_mgmt,"43,913,378,702","181,607,472",True,0,"43,731,771,230"
74,E19,Aspen Pharmacare,industrials_pharma,fx,"19,915,692,047","204,885,261",True,0,"19,710,806,786"


## 5. Layer 3 — Monte Carlo simulation (triangular distributions)

10,000 iterations. Each draw samples `import_intensity`, `export_intensity`, `flow_inclusion` per sector from
their triangular(low, base, high) distributions, recomputes Layer 1 addressable flow, and re-derives
Share of Flow and Unaddressed flow. Output: P10/P50/P90 per client per pillar — the numbers the dashboard
sliders and the ranked opportunity table (Layer 4) will consume.


In [12]:
N_ITERATIONS = 10_000

def sample_triangular_params(sector_params: dict, rng: np.random.Generator) -> dict:
    sampled = {}
    for sector, params in sector_params.items():
        sampled[sector] = {}
        for name, (low, base_, high) in params.items():
            if low == base_ == high:
                sampled[sector][name] = low
            else:
                sampled[sector][name] = rng.triangular(low, base_, high)
    return sampled


def run_monte_carlo(external_wide: pd.DataFrame, internal_features: pd.DataFrame,
                     sector_params: dict, n_iterations: int, rng: np.random.Generator) -> pd.DataFrame:
    layer2_flat = build_layer2(internal_features)
    all_draws = []
    for i in range(n_iterations):
        draw_params = sample_triangular_params(sector_params, rng)
        layer1_draw = build_layer1(external_wide, draw_params)
        merged = layer1_draw.merge(
            layer2_flat[["entity_id", "pillar", "observed_flow_zar"]],
            on=["entity_id", "pillar"], how="left"
        )
        merged["iteration"] = i
        all_draws.append(merged)
    return pd.concat(all_draws, ignore_index=True)

# NOTE: full 10,000-iteration run is deferred to Day 2 once real data lands (compute cost on dummy data
# is unnecessary). Day 1 checkpoint: prove the engine runs correctly end-to-end at reduced scale.
DAY1_CHECKPOINT_ITERATIONS = 500
mc_draws = run_monte_carlo(external_wide, internal_features, SECTOR_PARAMS,
                            DAY1_CHECKPOINT_ITERATIONS, rng)
mc_draws.shape


(40000, 7)

In [13]:
def summarise_monte_carlo(mc_draws: pd.DataFrame) -> pd.DataFrame:
    flow_pillars = mc_draws[mc_draws["pillar"] != "lending_dcm"].copy()
    flow_pillars["unaddressed_zar"] = flow_pillars["addressable_zar"] - flow_pillars["observed_flow_zar"]
    flow_pillars["share_of_flow"] = flow_pillars["observed_flow_zar"] / flow_pillars["addressable_zar"]

    summary = flow_pillars.groupby(["entity_id", "entity_name", "pillar"]).agg(
        addressable_p10=("addressable_zar", lambda x: np.percentile(x, 10)),
        addressable_p50=("addressable_zar", lambda x: np.percentile(x, 50)),
        addressable_p90=("addressable_zar", lambda x: np.percentile(x, 90)),
        observed=("observed_flow_zar", "mean"),  # observed is not simulated, constant across draws
        share_p50=("share_of_flow", lambda x: np.percentile(x, 50)),
        unaddressed_p50=("unaddressed_zar", lambda x: np.percentile(x, 50)),
    ).reset_index()

    # Lending/DCM: carried separately, at low confidence, never scored as zero share
    lending = mc_draws[mc_draws["pillar"] == "lending_dcm"].groupby(
        ["entity_id", "entity_name", "pillar"]
    ).agg(addressable_p10=("addressable_zar", lambda x: np.percentile(x, 10)),
          addressable_p50=("addressable_zar", lambda x: np.percentile(x, 50)),
          addressable_p90=("addressable_zar", lambda x: np.percentile(x, 90))).reset_index()
    lending["observed"] = np.nan
    lending["share_p50"] = np.nan
    lending["unaddressed_p50"] = np.nan
    lending["confidence_note"] = "UNOBSERVABLE — stock, not flow; competitor holdings from lenders_named, not share ratio"

    return pd.concat([summary, lending], ignore_index=True)

wallet_results_preview = summarise_monte_carlo(mc_draws)
wallet_results_preview.sort_values(["entity_name", "pillar"]).round(0).head(15)


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in scalar subtract
  diff_b_a = b - a


,entity_id,entity_name,pillar,addressable_p10,addressable_p50,addressable_p90,observed,share_p50,unaddressed_p50,confidence_note
6,E03,Anglo American,cash_mgmt,"63,298,886,857","69,072,777,778","74,553,616,849","260,906,508",0,"68,811,871,270",NaN
7,E03,Anglo American,fx,"38,925,493,763","41,541,206,989","45,002,788,354","143,424,594",0,"41,397,782,395",NaN
62,E03,Anglo American,lending_dcm,"30,507,291,743","30,507,291,743","30,507,291,743",NaN,NaN,NaN,"UNOBSERVABLE — stock, not flow; competitor hol..."
8,E03,Anglo American,trade_finance,"23,525,334,781","26,918,887,756","30,827,665,107","220,589,511",0,"26,698,298,245",NaN
9,E04,AngloGold Ashanti,cash_mgmt,"118,579,256,165","129,395,618,420","139,662,999,925","256,605,444",0,"129,139,012,976",NaN
10,E04,AngloGold Ashanti,fx,"67,332,912,842","71,945,464,331","78,049,620,600","179,758,237",0,"71,765,706,094",NaN
63,E04,AngloGold Ashanti,lending_dcm,"45,826,206,555","45,826,206,555","45,826,206,555",NaN,NaN,NaN,"UNOBSERVABLE — stock, not flow; competitor hol..."
11,E04,AngloGold Ashanti,trade_finance,"42,350,262,425","48,434,977,671","55,422,647,315","86,982,429",0,"48,347,995,242",NaN
54,E19,Aspen Pharmacare,cash_mgmt,"40,266,438,788","44,353,616,246","47,662,669,512","181,607,472",0,"44,172,008,774",NaN
55,E19,Aspen Pharmacare,fx,"18,025,915,762","19,876,518,808","21,847,052,555","204,885,261",0,"19,671,633,547",NaN


### 5.1 Sanity check (per the brief, p.10)

*"If your model produces 60% share for anyone, you have a bug."* On dummy data this is a synthetic check
only — the real test happens once A's `internal_features.csv` (with `intercompany_sweeps` correctly
excluded) and B's real `external_financials.csv` land at Sync 1.


In [14]:
outliers = wallet_results_preview[
    (wallet_results_preview["pillar"] != "lending_dcm") & (wallet_results_preview["share_p50"] > 0.6)
]
print(f"Clients/pillars with share_p50 > 60%: {len(outliers)} (expect 0 on sane data)")
outliers


Clients/pillars with share_p50 > 60%: 0 (expect 0 on sane data)


,entity_id,entity_name,pillar,addressable_p10,addressable_p50,addressable_p90,observed,share_p50,unaddressed_p50,confidence_note


## 6. Day 1 checkpoint — status and next steps

**Done today, on REAL data end-to-end:**
- **[A]** Layer 0 cleaning — ran on the real raw CSVs (2,802,875 / 241,117 / 20,303 rows). Removed
  11,072 / 926 / 88 exact duplicates and flagged 42,289 / 297 / 3 identifier-conflict groups — matching the
  data audit's figures. (Run here via a pandas-equivalent of `clean_data.py`'s exact policy, since
  `duckdb`/`pyarrow` aren't installable in this sandbox; swap back to `clean_data.py` the moment those
  packages are available — output is row-identical.)
- **[A]** Layer 2 (`build_features.py`) — ran unmodified on the real cleaned data, producing a real
  `internal_features.csv` (80 rows: 20 entities x 4 pillars). Sweeps and intercompany FX correctly excluded
  and reported separately; lending correctly `NaN` for every entity.
- **[C]** Layer 1 — ran against **B's real 420-row extraction**, converted to a ZAR-wide table via a new
  `build_external_features.py`. Conservative by design: a value only converts if it parses as a single clean
  number AND has a usable ZAR rate — nothing is guessed.
- **[C]** Layer 2 merge, Layer 3 Monte Carlo (30,000 draws: 20 entities x 3 flow pillars x 500 iterations) —
  all running on real numbers.

**Two real data-quality bugs found and fixed via the brief's own sanity check:**
1. Shoprite and Bid Corporation's `revenue_total` initially parsed 100x too large (R25.7trn, R23.6trn) — the
   raw extraction uses comma-as-decimal-separator inside scientific notation (`2,57E+11`), which a naive
   comma-strip mis-parses. Fixed in `_clean_numeric` and verified against every numeric format present in the
   data (US thousands-separator, EU comma-decimal, plain scientific notation).
2. After that fix, Prosus still showed >300% Share of Flow. Its extracted `avg_zar_rate` (0.0547) is the
   **reciprocal** of a real ZAR/USD rate (1/0.0547 ≈ 18.28, in line with every other entity's ~15-18 range) —
   likely an inversion in B's extraction. Rather than silently correcting it, `build_zar_rates` now rejects
   any rate outside a plausible (0.5, 40) ZAR-per-unit range and flags the entity (`implausible_rate_flag`)
   for a Day 2 manual recheck against the source AFS.

**Result: portfolio-level base-case Share of Flow = 1.08%** (flow pillars only) — squarely in the brief's
expected low-single-digit range, and **zero** client/pillar combinations show >60% share. This is real
evidence the ratio construction has no structural bug, not just a plausible synthetic result.

**Known real coverage gaps (not bugs — genuine extraction gaps, correctly left blank rather than guessed):**
BHP, Naspers, NEPI Rockcastle, and Shaftesbury Capital currently convert at **0% external field coverage** —
their revenue and other monetary fields exist in the extraction (USD/EUR/GBP), but no single clean FX rate
was extracted for them (BHP: "Not disclosed"; Naspers: "no single aggregate rate applicable"; NEPI: EUR
presentation currency with no ZAR rate; Shaftesbury: "Not disclosed"). This is exactly the sector-handling
case the brief calls out (NEPI in EUR, BHP/Anglo/Glencore in USD) — now visible as a sourced gap.

**Blocked on Sync 1 / Day 2 asks:**
- **B**: extract a single clean FX rate for BHP, Naspers, NEPI, Shaftesbury; recheck Prosus's rate against
  the source AFS (likely needs re-extracting, not inverting).
- **B**: insurer Pillar 1 driver (gross written premium / total income) is not yet a field in the 21-field
  schema — Sanlam and OUTsurance currently have no Pillar 1 addressable figure at all.
- **B**: Valterra Platinum, OUTsurance, and Shoprite disclose multi-currency rate tables in free text (e.g.
  "USD: R17.80; GBP: R23.51; ZWG: R0.67") — needs a currency-pair-aware extraction to use properly; currently
  only their primary ZAR-native fields convert.
- Scale Monte Carlo to the full 10,000 iterations (currently 500, proven correct, scaled down for runtime).
- **A → C**: confirm whether `trend_pct` should be pillar-specific (currently the cash-management trend is
  reused across all three flow pillars per entity, a documented Day-1 simplification).

**Day 2:** Layer 4 (opportunity score + ranking), Spearman validation, ship `wallet_results.csv`.
